# Classificação da geometria do pressing — GNN + stacking

## 1. Introdução e objetivo

As features escalares do freeze frame 360 (contagens, distâncias) descrevem o entorno do portador
da bola, mas **colapsar a configuração 2D dos jogadores em um único rótulo** (k-means / regras)
**perde informação** — é uma redução de dimensionalidade 2D→1D. Seguindo a sugestão do professor,
este notebook avalia a disposição espacial com uma **Graph Neural Network (GNN)**, que lê o grafo
dos jogadores sem esse colapso.

**Parte 1 — modelos da configuração espacial:**
- *Baseline A* — tipologia por regras (rótulo 1D)
- *Baseline B* — k-means com k variável (rótulo 1D)
- *GNN* — leitura 2D do grafo de jogadores

**Parte 2 — combinação por stacking:** a regressão logística de **contexto** (não-espacial) e a
GNN (geometria) são combinadas por um meta-modelo sobre as probabilidades previstas — **sem**
injetar rótulo na regressão. Compara-se a AUC de cada modelo isolado e do stacking.

## 2. Configuração, dados e engenharia de features

In [ ]:
import json, os, math, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
np.random.seed(42)

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, roc_curve, silhouette_score
from sklearn.model_selection import train_test_split

try:
    import torch
    import torch.nn as nn
    HAS_TORCH = True
    torch.manual_seed(42)
except Exception as e:
    HAS_TORCH = False
    print('torch indisponivel -> fallback DeepSets (MLP/sklearn). Detalhe:', e)

print('Setup pronto. HAS_TORCH =', HAS_TORCH)

In [ ]:
DATA_DIR = Path('../StatsBomb_2/data')
if not DATA_DIR.exists():
    DATA_DIR = Path('C:/Users/Abraao Ideapad3/Documents/Projetos/COE609/Analise_pressao_1/StatsBomb_2/data')
COMP_ID, SEASON_ID = 55, 43
RECOVERY_WINDOW = 10.0

def load_json(p):
    with open(p, encoding='utf-8') as fh:
        return json.load(fh)

matches = load_json(DATA_DIR / 'matches' / str(COMP_ID) / (str(SEASON_ID) + '.json'))
match_ids = sorted(m['match_id'] for m in matches)
parts, frames = [], {}
for mid in match_ids:
    edf = pd.json_normalize(load_json(DATA_DIR / 'events' / (str(mid) + '.json')), sep='_')
    edf['match_id'] = mid
    parts.append(edf)
    tp = DATA_DIR / 'three-sixty' / (str(mid) + '.json')
    if tp.exists():
        for fr in load_json(tp):
            frames[fr['event_uuid']] = fr
events = pd.concat(parts, ignore_index=True)
print('jogos:', len(match_ids), '| eventos:', len(events), '| freeze frames:', len(frames))

In [ ]:
# Variavel resposta: recovered (recuperacao em ate 10s pelo mesmo time)
def ts_to_seconds(ts):
    hh, mm, ss = ts.split(':')
    return int(hh) * 3600 + int(mm) * 60 + float(ss)

events['t'] = events['timestamp'].apply(ts_to_seconds)
pressures = events[events['type_name'] == 'Pressure'].copy()

if 'ball_recovery_recovery_failure' in events.columns:
    rec_fail = events['ball_recovery_recovery_failure'].fillna(False)
else:
    rec_fail = pd.Series(False, index=events.index)
duel_out = events.get('duel_outcome_name', pd.Series('', index=events.index)).fillna('')
rec_mask = ((events['type_name'].eq('Ball Recovery') & ~rec_fail)
            | events['type_name'].eq('Interception')
            | (events['type_name'].eq('Duel') & duel_out.str.contains('Won|Success')))
rec = events.loc[rec_mask, ['match_id', 'period', 'team_id', 't']]
L = pressures[['id', 'match_id', 'period', 'team_id', 't']].rename(columns={'t': 'tp'}).sort_values('tp')
R = rec.rename(columns={'t': 'tr'}).sort_values('tr')
mm = pd.merge_asof(L, R, left_on='tp', right_on='tr', by=['match_id', 'period', 'team_id'],
                   direction='forward', tolerance=RECOVERY_WINDOW)
mm['recovered'] = mm['tr'].notna().astype(int)
pressures = pressures.merge(mm[['id', 'recovered']], on='id', how='left')
pressures['recovered'] = pressures['recovered'].fillna(0).astype(int)
pressures = pressures.reset_index(drop=True)
print('pressoes:', len(pressures), '| taxa de recuperacao: %.3f' % pressures['recovered'].mean())

In [ ]:
# Features de contexto (nao-espaciais) para o modelo logistico do stacking
xy = pressures['location'].apply(lambda v: v[:2] if isinstance(v, list) and len(v) >= 2 else [np.nan, np.nan])
pressures['x'] = xy.apply(lambda v: v[0])
pressures['y'] = xy.apply(lambda v: v[1])
pressures['dist_to_goal'] = np.sqrt((120 - pressures['x']) ** 2 + (40 - pressures['y']) ** 2)
pressures['zone'] = pd.cut(pressures['x'], [-0.1, 40, 80, 120.1], labels=['Def', 'Mid', 'Att']).astype(str)
pressures['duration'] = pressures['duration'].fillna(0.0)
print(pressures[['x', 'y', 'dist_to_goal', 'duration']].describe().round(2))

### 2.1 Primitivas geométricas (para os baselines)

Para cada pressão, do freeze frame 360. **Portador** = jogador do time pressionado (`teammate==False`)
mais próximo da bola; **adversários** do portador = `teammate==True` (time pressionante); **apoio** =
demais `teammate==False`. As coordenadas estão no referencial do pressionante (ataca →x=120), logo a
**frente do portador** é x decrescente (reorientamos para frente = +X). Primitivas: densidade radial
(5/10/15 m), direção (cobertura em arco, maior vão livre, vão livre à frente) e relação apoio×adversários.

In [ ]:
def primitives(row):
    keys = ['n_adv_5m','n_adv_10m','n_adv_15m','n_sup_5m','n_sup_10m','n_sup_15m',
            'adv_arc_coverage','largest_free_angle','free_lane_forward','nearest_support_dist',
            'support_lane_blocked','n_free_support','adv_between_count','carrier_x','carrier_y','has_frame']
    nan = {k: np.nan for k in keys}; nan['has_frame'] = 0
    fr = frames.get(row['id'])
    if fr is None or not isinstance(row['location'], list):
        return pd.Series(nan)
    ball = np.array(row['location'][:2], float)
    advs, sups = [], []
    for p in fr['freeze_frame']:
        xyp = np.array(p['location'], float)
        (advs if p.get('teammate') else sups).append(xyp)
    if len(sups) == 0:
        return pd.Series(nan)
    sups = np.array(sups); advs = np.array(advs) if advs else np.zeros((0, 2))
    ci = int(np.argmin(np.linalg.norm(sups - ball, axis=1)))
    carrier = sups[ci]
    support = np.delete(sups, ci, axis=0)
    def rel(P):
        if len(P) == 0: return P.reshape(0, 2)
        d = P - carrier
        return np.column_stack([-d[:, 0], d[:, 1]])   # frente = +X
    adv_r, sup_r = rel(advs), rel(support)
    adv_d = np.linalg.norm(adv_r, axis=1) if len(adv_r) else np.array([])
    sup_d = np.linalg.norm(sup_r, axis=1) if len(sup_r) else np.array([])
    res = {}
    for r in (5, 10, 15):
        res['n_adv_%dm' % r] = int((adv_d <= r).sum())
        res['n_sup_%dm' % r] = int((sup_d <= r).sum())
    near = adv_r[adv_d <= 10] if len(adv_r) else adv_r
    angs = []
    if len(near):
        angs = sorted(((np.degrees(np.arctan2(near[:, 1], near[:, 0])) + 360) % 360).tolist())
    res['adv_arc_coverage'] = len(set(int(a // 45) for a in angs)) / 8.0
    if len(angs) == 0:
        res['largest_free_angle'] = 360.0; free_mid = 0.0
    elif len(angs) == 1:
        res['largest_free_angle'] = 360.0; free_mid = (angs[0] + 180) % 360
    else:
        gaps = [((angs[(i + 1) % len(angs)] - angs[i]) % 360, i) for i in range(len(angs))]
        gap, i = max(gaps); free_mid = (angs[i] + gap / 2) % 360
        res['largest_free_angle'] = gap
    res['free_lane_forward'] = int(math.cos(math.radians(free_mid)) > 0)
    cos15 = math.cos(math.radians(15))
    def blocked(target):
        tt = np.linalg.norm(target)
        if tt == 0 or len(adv_r) == 0: return False
        for a in adv_r:
            da = np.linalg.norm(a)
            if da < tt and np.dot(a, target) / (da * tt + 1e-9) > cos15:
                return True
        return False
    if len(support) == 0:
        res['nearest_support_dist'] = np.nan; res['support_lane_blocked'] = np.nan
        res['n_free_support'] = 0; res['adv_between_count'] = 0
    else:
        res['nearest_support_dist'] = float(sup_d.min())
        res['support_lane_blocked'] = int(blocked(sup_r[int(np.argmin(sup_d))]))
        res['n_free_support'] = int(sum(not blocked(s) for s in sup_r))
        res['adv_between_count'] = int(sum(blocked(s) for s in sup_r))
    res['carrier_x'] = float(carrier[0]); res['carrier_y'] = float(carrier[1]); res['has_frame'] = 1
    return pd.Series(res)

prim_df = pressures.apply(primitives, axis=1)
pressures = pd.concat([pressures, prim_df], axis=1)
PRIM = ['n_adv_5m','n_adv_10m','n_adv_15m','n_sup_5m','n_sup_10m','n_sup_15m','adv_arc_coverage',
        'largest_free_angle','free_lane_forward','nearest_support_dist','support_lane_blocked',
        'n_free_support','adv_between_count']
print('cobertura de freeze frame: %.3f' % pressures['has_frame'].mean())
print(pressures.loc[pressures['has_frame'] == 1, PRIM].describe().round(2).T)

### 2.2 Construção dos grafos (para a GNN)

Cada pressão vira um grafo: **nós** = jogadores visíveis a ≤30 m do portador (no máximo os 24 mais
próximos); **features de nó** = posição reorientada (frente = +X), distância ao portador, distância
à bola e one-hot do papel (portador / adversário / apoio); **arestas** = pares a ≤10 m (com
self-loops), com adjacência normalizada simétrica. Rótulo do grafo = `recovered`.

In [ ]:
NEIGH_R, LOCAL_R, NMAX, FEAT_DIM = 10.0, 30.0, 24, 7

def build_graph(row):
    fr = frames.get(row['id'])
    if fr is None or row['has_frame'] != 1 or not isinstance(row['location'], list):
        return None
    ball = np.array(row['location'][:2], float)
    carrier = np.array([row['carrier_x'], row['carrier_y']], float)
    nodes, is_adv = [], []
    for p in fr['freeze_frame']:
        xyp = np.array(p['location'], float)
        if np.linalg.norm(xyp - carrier) <= LOCAL_R:
            nodes.append(xyp); is_adv.append(1 if p.get('teammate') else 0)
    if len(nodes) < 2:
        return None
    nodes = np.array(nodes); is_adv = np.array(is_adv)
    order = np.argsort(np.linalg.norm(nodes - carrier, axis=1))[:NMAX]
    nodes, is_adv = nodes[order], is_adv[order]
    n = len(nodes)
    role = np.zeros(n, int)        # 0 = adversario (pressionante)
    role[is_adv == 0] = 1          # 1 = apoio (pressionado)
    role[0] = 2                    # 0-esimo (mais proximo do portador) = portador
    d = nodes - carrier
    fwd = np.column_stack([-d[:, 0], d[:, 1]])
    dist_c = np.linalg.norm(fwd, axis=1)
    dist_b = np.linalg.norm(nodes - ball, axis=1)
    onehot = np.eye(3)[role]
    feat = np.column_stack([fwd / 30.0, (dist_c / 30.0)[:, None], (dist_b / 30.0)[:, None], onehot]).astype(np.float32)
    Dm = np.linalg.norm(nodes[:, None, :] - nodes[None, :, :], axis=2)
    A = (Dm <= NEIGH_R).astype(float)
    np.fill_diagonal(A, 1.0)
    dinv = 1.0 / np.sqrt(A.sum(1, keepdims=True))
    A = (A * dinv * dinv.T).astype(np.float32)
    return feat, A, n

graphs, valid_idx = [], []
for i, row in pressures.iterrows():
    g = build_graph(row)
    if g is not None:
        graphs.append(g); valid_idx.append(i)
valid_idx = np.array(valid_idx)

X = np.zeros((len(graphs), NMAX, FEAT_DIM), np.float32)
A = np.zeros((len(graphs), NMAX, NMAX), np.float32)
Mk = np.zeros((len(graphs), NMAX), np.float32)
for gi, (feat, a, n) in enumerate(graphs):
    X[gi, :n] = feat; A[gi, :n, :n] = a; Mk[gi, :n] = 1.0
y = pressures.loc[valid_idx, 'recovered'].to_numpy().astype(np.float32)
print('grafos:', len(graphs), '| X:', X.shape, '| taxa y: %.3f' % y.mean())

# Parte 1 — Modelos da configuração espacial

## 3. Baseline A — tipologia por regras (rótulo 1D)

In [ ]:
# Limiares calibrados pela distribuicao (mediana do maior vao livre)
FREE_MED = float(pressures.loc[pressures['has_frame'] == 1, 'largest_free_angle'].median())

def classify_rule(r):
    if r['has_frame'] != 1:
        return 'Sem_frame'
    if r['n_adv_10m'] <= 1:
        return 'Livre'                  # poucos adversarios ate 10 m
    if r['n_adv_5m'] >= 2:
        return 'Cercado'                # dois+ adversarios colados (<=5 m)
    if r['largest_free_angle'] >= FREE_MED and r['free_lane_forward'] == 1:
        return 'Saida_a_frente'         # vao livre amplo apontando para a frente
    if r['n_free_support'] >= 1 and r['nearest_support_dist'] <= 12:
        return 'Com_apoio'              # apoio proximo com linha de passe livre
    return 'Sem_saida'                  # pressionado, sem vao a frente nem apoio

pressures['ctx_rule'] = pressures.apply(classify_rule, axis=1)

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0, c - h), min(1, c + h))

sub = pressures[pressures['has_frame'] == 1]
tab = sub.groupby('ctx_rule')['recovered'].agg(['size', 'mean'])
tab['ic_inf'], tab['ic_sup'] = zip(*[wilson(int(m * s), int(s)) for s, m in zip(tab['size'], tab['mean'])])
ct = pd.crosstab(sub['ctx_rule'], sub['recovered'])
chi2, pchi, dof, _ = stats.chi2_contingency(ct)
cramer = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
print('Qui-quadrado p = %.3g | Cramers V = %.3f' % (pchi, cramer))
print(tab.round(3))

In [ ]:
# Visualizacao Baseline A: distribuicao e taxa de recuperacao por classe
glob = sub['recovered'].mean()
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
t = tab.sort_values('mean')
ax[0].bar(t.index, t['size'], color='#1565c0'); ax[0].set_title('Distribuicao das classes (regras)')
ax[0].tick_params(axis='x', rotation=30)
yerr = [t['mean'] - t['ic_inf'], t['ic_sup'] - t['mean']]
ax[1].bar(t.index, t['mean'] * 100, yerr=np.array(yerr) * 100, capsize=4, color='#2e7d32')
ax[1].axhline(glob * 100, color='red', ls='--'); ax[1].set_ylabel('% recuperacao')
ax[1].set_title('Taxa de recuperacao por classe (linha = media geral)')
ax[1].tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

In [ ]:
# Mini-diagramas: uma pressao tipica de cada classe (geometria ao redor do portador)
def draw_config(row, ax):
    fr = frames.get(row['id'])
    carrier = np.array([row['carrier_x'], row['carrier_y']])
    ax.add_patch(plt.Rectangle((0, 0), 120, 80, fill=False, ec='gray'))
    for p in fr['freeze_frame']:
        xyp = p['location']
        if p.get('teammate'):
            ax.scatter(*xyp, c='#c62828', s=25)          # adversario (pressionante)
        else:
            ax.scatter(*xyp, c='#1565c0', s=25)          # apoio (pressionado)
    ax.scatter(*carrier, c='black', marker='*', s=160)   # portador
    ax.set_xlim(carrier[0] - 25, carrier[0] + 25); ax.set_ylim(carrier[1] - 25, carrier[1] + 25)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect('equal')

classes = [c for c in tab.index if c != 'Sem_frame']
fig, axes = plt.subplots(1, len(classes), figsize=(3.2 * len(classes), 3.4))
if len(classes) == 1: axes = [axes]
for ax, cl in zip(axes, classes):
    ex = sub[sub['ctx_rule'] == cl]
    draw_config(ex.iloc[0], ax)
    ax.set_title(cl, fontsize=9)
fig.suptitle('Exemplo de configuracao por classe  (preto=portador, azul=apoio, vermelho=adversario)', fontsize=10)
plt.tight_layout(); plt.show()

## 4. Baseline B — k-means (rótulo 1D), k variável

In [ ]:
PRIMV = pressures.loc[valid_idx, PRIM].astype(float)
PRIMV = PRIMV.fillna(PRIMV.median()).to_numpy()
Z = StandardScaler().fit_transform(PRIMV)

ks = range(2, 9)
inertias, sils = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Z)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(Z, km.labels_, sample_size=4000, random_state=42))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(list(ks), inertias, '-o'); ax[0].set_title('Cotovelo (inercia)'); ax[0].set_xlabel('k')
ax[1].plot(list(ks), sils, '-o', color='#ef6c00'); ax[1].set_title('Silhueta media'); ax[1].set_xlabel('k')
plt.tight_layout(); plt.show()
print('silhueta por k:', dict(zip(ks, np.round(sils, 3))))

In [ ]:
K_BEST = int(list(ks)[int(np.argmax(sils))])
km = KMeans(n_clusters=K_BEST, n_init=10, random_state=42).fit(Z)
KMLAB = km.labels_
prof = pd.DataFrame(Z, columns=PRIM); prof['cluster'] = KMLAB
prof_mean = prof.groupby('cluster').mean()
yv = pd.Series(y)
rate = yv.groupby(KMLAB).mean(); size = yv.groupby(KMLAB).size()

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
im = ax[0].imshow(prof_mean.T, aspect='auto', cmap='coolwarm', vmin=-1.5, vmax=1.5)
ax[0].set_yticks(range(len(PRIM))); ax[0].set_yticklabels(PRIM, fontsize=7)
ax[0].set_xticks(range(K_BEST)); ax[0].set_xlabel('cluster')
ax[0].set_title('Perfil dos clusters (z-score das primitivas)'); fig.colorbar(im, ax=ax[0])
ax[1].bar(range(K_BEST), rate * 100, color='#2e7d32')
ax[1].axhline(y.mean() * 100, color='red', ls='--'); ax[1].set_xlabel('cluster')
ax[1].set_ylabel('% recuperacao'); ax[1].set_title('Recuperacao por cluster (k=%d)' % K_BEST)
plt.tight_layout(); plt.show()
print('k escolhido:', K_BEST, '| tamanhos:', size.to_dict())

In [ ]:
# Projecao PCA 2D colorida por cluster
pc = PCA(n_components=2, random_state=42).fit_transform(Z)
fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(pc[:, 0], pc[:, 1], c=KMLAB, cmap='tab10', s=6, alpha=0.4)
ax.set_title('PCA das primitivas (cor = cluster k-means)'); ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.tight_layout(); plt.show()

## 5. GNN — leitura 2D da configuração (sem colapso em rótulo)

GCN denso e compacto (2 camadas de message passing + pooling global média/máx + MLP) que prevê
P(recovered) direto do grafo. Se `torch` não estiver disponível, usa-se um fallback
permutation-invariant (DeepSets: MLP sobre as features de nó agregadas por média+máx).

In [ ]:
if HAS_TORCH:
    class DenseGCN(nn.Module):
        def __init__(self, fdim, hid=32, nlayers=2, p=0.2):
            super().__init__()
            self.lins = nn.ModuleList([nn.Linear(fdim if i == 0 else hid, hid) for i in range(nlayers)])
            self.head = nn.Sequential(nn.Linear(2 * hid, hid), nn.ReLU(), nn.Dropout(p), nn.Linear(hid, 1))
        def forward(self, X, A, M):
            H = X
            for lin in self.lins:
                H = torch.relu(torch.bmm(A, lin(H))) * M.unsqueeze(-1)
            mm = M.unsqueeze(-1)
            mean = (H * mm).sum(1) / mm.sum(1).clamp(min=1)
            mx = H.masked_fill(mm == 0, -1e9).max(1).values
            return self.head(torch.cat([mean, mx], 1)).squeeze(-1)

    Xt, At, Mt, yt = (torch.tensor(X), torch.tensor(A), torch.tensor(Mk), torch.tensor(y))

    def train_gnn(tr, va, epochs=60, bs=256, lr=1e-3):
        model = DenseGCN(FEAT_DIM)
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
        lossf = nn.BCEWithLogitsLoss()
        best_auc, best_state, hist = -1, None, []
        for ep in range(epochs):
            model.train(); perm = np.random.permutation(tr)
            for s in range(0, len(perm), bs):
                b = torch.as_tensor(perm[s:s + bs], dtype=torch.long)
                opt.zero_grad()
                loss = lossf(model(Xt[b], At[b], Mt[b]), yt[b]); loss.backward(); opt.step()
            model.eval()
            with torch.no_grad():
                vb = torch.as_tensor(va, dtype=torch.long)
                auc = roc_auc_score(y[va], torch.sigmoid(model(Xt[vb], At[vb], Mt[vb])).numpy())
            hist.append(auc)
            if auc > best_auc:
                best_auc = auc; best_state = {k: v.clone() for k, v in model.state_dict().items()}
        model.load_state_dict(best_state)
        return model, hist

    def gnn_predict(model, idx):
        model.eval()
        with torch.no_grad():
            ib = torch.as_tensor(idx, dtype=torch.long)
            return torch.sigmoid(model(Xt[ib], At[ib], Mt[ib])).numpy()

def geom_pooled(idx):
    out = []
    for i in idx:
        n = int(Mk[i].sum()); nf = X[i, :max(n, 1)]
        out.append(np.concatenate([nf.mean(0), nf.max(0)]))
    return np.array(out)

def fit_geom(tr, va):
    if HAS_TORCH:
        model, hist = train_gnn(tr, va)
        return ('gnn', model, hist)
    clf = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=300, random_state=42)
    clf.fit(geom_pooled(tr), y[tr])
    return ('mlp', clf, [])

def predict_geom(model, idx):
    if model[0] == 'gnn':
        return gnn_predict(model[1], idx)
    return model[1].predict_proba(geom_pooled(idx))[:, 1]

print('modelo de geometria:', 'GNN (torch)' if HAS_TORCH else 'DeepSets/MLP (sklearn)')

In [ ]:
# Splits compartilhados (posicoes 0..Nvalid-1): TRAIN / STACK / TEST + VAL para early stopping
pos = np.arange(len(valid_idx))
tr, tmp = train_test_split(pos, test_size=0.40, stratify=y, random_state=42)
stack_i, test_i = train_test_split(tmp, test_size=0.50, stratify=y[tmp], random_state=42)
tr2, val_i = train_test_split(tr, test_size=0.20, stratify=y[tr], random_state=42)
print('train:', len(tr2), 'val:', len(val_i), 'stack:', len(stack_i), 'test:', len(test_i))

geom = fit_geom(tr2, val_i)
if geom[0] == 'gnn' and len(geom[2]):
    plt.figure(figsize=(6, 3.2)); plt.plot(geom[2], '-o', ms=3)
    plt.title('AUC de validacao por epoca (GNN)'); plt.xlabel('epoca'); plt.ylabel('AUC')
    plt.tight_layout(); plt.show()

In [ ]:
# Demonstracao da perda de informacao: rotulo 1D (regras / k-means) vs GNN
def label_rate_predict(labels, tr_pos, te_pos):
    s = pd.Series(y[tr_pos]).groupby(labels[tr_pos]).mean()
    g = y[tr_pos].mean()
    return np.array([s.get(l, g) for l in labels[te_pos]])

RULEV = pressures.loc[valid_idx, 'ctx_rule'].to_numpy()
auc_rule = roc_auc_score(y[test_i], label_rate_predict(RULEV, tr, test_i))
auc_km = roc_auc_score(y[test_i], label_rate_predict(KMLAB, tr, test_i))
p_geom_test = predict_geom(geom, test_i)
auc_geom = roc_auc_score(y[test_i], p_geom_test)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
names = ['Regras (1D)', 'K-means (1D)', 'GNN (2D)']
vals = [auc_rule, auc_km, auc_geom]
ax[0].bar(names, vals, color=['#b0bec5', '#ef6c00', '#2e7d32'])
ax[0].set_ylim(0.5, max(vals) + 0.03); ax[0].set_ylabel('AUC (teste)')
ax[0].set_title('Discriminacao: rotulo 1D vs GNN 2D')
for i, v in enumerate(vals):
    ax[0].text(i, v + 0.002, '%.3f' % v, ha='center')
for nm, sc in [('Regras', label_rate_predict(RULEV, tr, test_i)),
               ('K-means', label_rate_predict(KMLAB, tr, test_i)), ('GNN', p_geom_test)]:
    fpr, tpr, _ = roc_curve(y[test_i], sc); ax[1].plot(fpr, tpr, label=nm)
ax[1].plot([0, 1], [0, 1], 'k--', lw=0.8); ax[1].legend(); ax[1].set_title('Curvas ROC (teste)')
ax[1].set_xlabel('FPR'); ax[1].set_ylabel('TPR')
plt.tight_layout(); plt.show()
print('AUC -> regras: %.3f | k-means: %.3f | GNN: %.3f' % (auc_rule, auc_km, auc_geom))

# Parte 2 — Combinação por stacking

## 6. Logística de contexto + meta-modelo

**Modelo 1 (contexto):** regressão logística sobre features não-espaciais (posição no campo, minuto,
duração, terço). **Modelo 2 (geometria):** a GNN. O **meta-modelo** (logística) combina as
probabilidades previstas dos dois — treinado no conjunto STACK e avaliado no TESTE. Sem injetar
rótulo de classificação na regressão.

In [ ]:
def context_matrix(idx_labels):
    s = pressures.loc[idx_labels]
    z = pd.get_dummies(s['zone'])
    for c in ['Def', 'Mid', 'Att']:
        if c not in z: z[c] = 0
    M_ = np.column_stack([s['x'], s['y'], s['dist_to_goal'], s['minute'], s['duration'],
                          z['Def'], z['Mid'], z['Att']]).astype(float)
    return np.nan_to_num(M_, nan=0.0)

CTX = context_matrix(valid_idx)
ctx_scaler = StandardScaler().fit(CTX[tr])
ctx_model = LogisticRegression(max_iter=1000).fit(ctx_scaler.transform(CTX[tr]), y[tr])

def ctx_predict(idx):
    return ctx_model.predict_proba(ctx_scaler.transform(CTX[idx]))[:, 1]

p_ctx_stack, p_geom_stack = ctx_predict(stack_i), predict_geom(geom, stack_i)
p_ctx_test = ctx_predict(test_i)
meta = LogisticRegression(max_iter=1000).fit(
    np.column_stack([p_ctx_stack, p_geom_stack]), y[stack_i])
p_meta_test = meta.predict_proba(np.column_stack([p_ctx_test, p_geom_test]))[:, 1]

auc_ctx = roc_auc_score(y[test_i], p_ctx_test)
auc_geom_t = roc_auc_score(y[test_i], p_geom_test)
auc_meta = roc_auc_score(y[test_i], p_meta_test)
print('AUC (teste) -> contexto: %.3f | geometria: %.3f | stacking: %.3f' % (auc_ctx, auc_geom_t, auc_meta))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for nm, sc in [('Contexto', p_ctx_test), ('Geometria', p_geom_test), ('Stacking', p_meta_test)]:
    fpr, tpr, _ = roc_curve(y[test_i], sc); ax[0].plot(fpr, tpr, label='%s (AUC=%.3f)' % (nm, roc_auc_score(y[test_i], sc)))
ax[0].plot([0, 1], [0, 1], 'k--', lw=0.8); ax[0].legend(); ax[0].set_title('ROC no teste')
ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR')
sc1 = ax[1].scatter(p_ctx_test, p_geom_test, c=y[test_i], cmap='coolwarm', s=10, alpha=0.5)
ax[1].set_xlabel('P(recuperar) - contexto'); ax[1].set_ylabel('P(recuperar) - geometria')
ax[1].set_title('Complementaridade dos dois sinais'); fig.colorbar(sc1, ax=ax[1], label='recovered')
plt.tight_layout(); plt.show()

print('Coeficientes do meta-modelo [contexto, geometria]:', np.round(meta.coef_[0], 3))

## 7. Conclusão

- **Perda de informação (Seção 5):** comparar a AUC dos rótulos 1D (regras, k-means) com a da GNN
  2D. Se a GNN discrimina melhor, confirma-se empiricamente o ponto do professor — colapsar a
  geometria em um rótulo único descarta sinal preditivo.
- **Stacking (Seção 6):** se a AUC do stacking supera a do contexto e a da geometria isolados, os
  dois modelos são **complementares** — a geometria acrescenta informação que o contexto
  não-espacial não capta, e vice-versa.
- A regressão logística de contexto permanece como o modelo **interpretável**; a GNN entra como
  componente espacial, combinada por stacking — sem comprometer a interpretabilidade da logística
  nem injetar rótulos nela.

**Limitações:** freeze frame só vê jogadores no quadro; a bola é aproximada pela posição do evento
de pressão; a GNN é compacta e treinada em um único split (k-fold tornaria a estimativa mais
robusta); os limiares das regras são heurísticos e podem ser calibrados por quantis.